In [1]:
print(123)

123


In [2]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

('To run Ollama locally:\n\n1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system:\n   - macOS: download the `.pkg` and install it\n   - Windows: download the `.msi` and install it\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test that the local server is running, you can also run:\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models": [...]}\n```', 742, 179)


In [7]:
answer

('To run Ollama locally:\n\n1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system:\n   - macOS: download the `.pkg` and install it\n   - Windows: download the `.msi` and install it\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nTo test that the local server is running, you can also run:\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models": [...]}\n```',
 742,
 179)

In [8]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

('The FAQ context only explains how to run the **course locally**, not how to run **Olama** specifically.\n\nIf you mean running things locally in general, the course says you can do that if you’re comfortable setting up:\n\n- Python\n- `uv`\n- Jupyter\n- Docker\n- any other tools needed for the module\n\nIt also says to document your setup and keep it reproducible.\n\nIf you meant a specific local setup for **Olama**, that isn’t covered in the provided context.', 885, 107)


In [14]:
messages = [
     {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': 'How do I run Olama locally?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)



AttributeError: 'list' object has no attribute 'name'

In [21]:
response.output[0].type

'message'

In [10]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    
)

In [13]:
print(response)

Response(id='resp_02c8b9b6fa990fb7006a3bd5a2f95881a0821566871553d3a6', created_at=1782306210.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_02c8b9b6fa990fb7006a3bd5a36b4481a09cb11606f4371ce5', content=[ResponseOutputText(annotations=[], text='Use the local startup command:\n\n```bash\nollama serve\n```\n\nThen connect to it at:\n\n```text\nhttp://127.0.0.1:11434\n```\n\nIf you want, I can also help you verify it’s running.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1782306211.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='24h', reasoning=Reasoning(effort='none', gene

In [14]:
print(response.output_text)

Use the local startup command:

```bash
ollama serve
```

Then connect to it at:

```text
http://127.0.0.1:11434
```

If you want, I can also help you verify it’s running.


In [15]:
response.output

[ResponseOutputMessage(id='msg_02c8b9b6fa990fb7006a3bd5a36b4481a09cb11606f4371ce5', content=[ResponseOutputText(annotations=[], text='Use the local startup command:\n\n```bash\nollama serve\n```\n\nThen connect to it at:\n\n```text\nhttp://127.0.0.1:11434\n```\n\nIf you want, I can also help you verify it’s running.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [16]:
len(response.output)

1

In [17]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [18]:
print(response.output_text)

In [19]:
len(response.output)

1

In [20]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama locally install start use localhost FAQ"}', call_id='call_ufs53SNzIitHUtftD3hSCGd8', name='search', type='function_call', id='fc_04a724a608cdb4ef006a3bd5d6079c819ca2d65bfaaf1ea1e7', namespace=None, status='completed')]

In [21]:
# to remove square brackets
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama locally install start use localhost FAQ"}', call_id='call_ufs53SNzIitHUtftD3hSCGd8', name='search', type='function_call', id='fc_04a724a608cdb4ef006a3bd5d6079c819ca2d65bfaaf1ea1e7', namespace=None, status='completed')

In [22]:
call = response.output[0]

In [23]:
call

ResponseFunctionToolCall(arguments='{"query":"Olama locally run Ollama locally install start use localhost FAQ"}', call_id='call_ufs53SNzIitHUtftD3hSCGd8', name='search', type='function_call', id='fc_04a724a608cdb4ef006a3bd5d6079c819ca2d65bfaaf1ea1e7', namespace=None, status='completed')

In [24]:
call.arguments

'{"query":"Olama locally run Ollama locally install start use localhost FAQ"}'

In [26]:
import json

args = json.loads(call.arguments)
args

NameError: name 'call' is not defined

In [27]:
call.name

NameError: name 'call' is not defined

In [28]:
results = search(**args)

NameError: name 'args' is not defined

In [29]:
result_json = json.dumps(results, indent=2)
print(result_json)

NameError: name 'results' is not defined

In [30]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}
function_call_output

NameError: name 'call' is not defined

In [31]:
messages.append(call)

NameError: name 'call' is not defined

In [32]:
messages.append(function_call_output)

NameError: name 'function_call_output' is not defined

In [33]:
print(messages)

[{'role': 'developer', 'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"}, {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join"}', call_id='call_TJjIoKZplpLpthB0NxaqkCdk', name='search', type='function_call', id='fc_0bc68cedb9be6a9b006a3bdb4dc158819d8ae5456a81ce289c', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course FAQ join discovered course can I enroll now"}', call_id='call_l

In [34]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

BadRequestError: Error code: 400 - {'error': {'message': 'No tool output found for function call call_TJjIoKZplpLpthB0NxaqkCdk.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}

In [35]:
print(response.output_text)

In [36]:
response.output[0]

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join"}', call_id='call_TJjIoKZplpLpthB0NxaqkCdk', name='search', type='function_call', id='fc_0bc68cedb9be6a9b006a3bdb4dc158819d8ae5456a81ce289c', namespace=None, status='completed')

In [37]:
response.output[0].type

'function_call'

In [38]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(166, 85)

In [39]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [40]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [41]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [42]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [44]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join"}', call_id='call_n7hL42fc3fW0SBB34DDMU9Mf', name='search', type='function_call', id='fc_0d27212b21ba657f006a3bdb83b744819c8c9b815c40774b5f', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered the course can I join"}', call_id='call_Kl41tXeMEfL4rT95USL0oGEL', name='search', type='function_call', id='fc_0d27212b21ba657f006a3bdb83b758819cb5a06cb175a794a2', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"auditing joining after course start enrollment FAQ"}', call_id='call_jw9tnQOk1xoFLKS45fQWkvs9', name='search', type='function_call', id='fc_0d27212b21ba657f006a3bdb83b76c819cb4147cb8fc99c1cf', namespace=None, status='completed')]

In [43]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered late enrollment can I join"}
function_call: search {"query":"course enrollment late join discovered the course can I join"}
function_call: search {"query":"auditing joining after course start enrollment FAQ"}


In [46]:
print(messages)

[{'role': 'developer', 'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"}, {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join"}', call_id='call_n7hL42fc3fW0SBB34DDMU9Mf', name='search', type='function_call', id='fc_0d27212b21ba657f006a3bdb83b744819c8c9b815c40774b5f', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course enrollment late join discovered the course can I join"}', call_id='call_Kl41tXeMEfL4rT95

In [32]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break


iteration #1...
function_call: search {"query":"join the course enrollment can I join course discovered just now"}
function_call: search {"query":"course enrollment late join discovered course can I still join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

You can start learning right away, and if you want a certificate, make sure to submit your project while submissions are still open. Homework is not mandatory, but it’s recommended.

If you want, I can also help you figure out how to get started with the course materials. Are there other areas you want to explore?


In [33]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [34]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [35]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course late discovered can I join course enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

One important caveat: if you want a certificate, you need to submit your project while submissions are still open. If you’re just looking to learn, you can start anytime.

If you want, I can also help you find the course materials, schedule, or how to get started.


In [36]:
result

'Yes — you can still join the course.\n\nOne important caveat: if you want a certificate, you need to submit your project while submissions are still open. If you’re just looking to learn, you can start anytime.\n\nIf you want, I can also help you find the course materials, schedule, or how to get started.'

In [37]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen gambit"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry related to “queen gambit,” so it looks like this is off-topic for the course.

If you meant something course-related, feel free to rephrase it, and I can look it up. Are there other areas you want to explore?


In [39]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [40]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [41]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [42]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [43]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [44]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [45]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [46]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [47]:
result.cost

CostInfo(input_cost=Decimal('0.00269325'), output_cost=Decimal('0.0012465'), total_cost=Decimal('0.00393975'))

In [48]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local setup Ollama course FAQ"}', call_

In [49]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [ ]:
runner.run();